# Task 4 — Classifier Comparison: TF-IDF Baseline vs BERT+LoRA
**AIML339 Email Thread Triage**

This notebook compares the two classifiers on the BC3 test set (37 emails, 6 threads).
Both models were evaluated on the same split and the same 6 multi-label categories.

Predictions are loaded from:
- `results/bc3_baseline_test_predictions.csv` (TF-IDF + OneVsRestClassifier)
- `results/bc3_bert_lora_test_predictions.csv` (bert-base-uncased + LoRA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss

LABEL_NAMES = ["Request", "Propose", "Commit", "Meeting", "Subjective", "Informative"]

baseline = pd.read_csv("../results/bc3_baseline_test_predictions.csv")
bert     = pd.read_csv("../results/bc3_bert_lora_test_predictions.csv")

# Build true and pred matrices for each model
y_true     = baseline[[f"true_{l}" for l in LABEL_NAMES]].values
y_baseline = baseline[[f"pred_{l}" for l in LABEL_NAMES]].values
y_bert     = bert[[f"pred_{l}" for l in LABEL_NAMES]].values

print(f"Test emails: {len(y_true)}")

## Overall metrics

In [ ]:
def overall_metrics(y_true, y_pred, name):
    return {
        "Model":        name,
        "Micro F1":     round(f1_score(y_true, y_pred, average="micro", zero_division=0), 3),
        "Macro F1":     round(f1_score(y_true, y_pred, average="macro", zero_division=0), 3),
        "Precision":    round(precision_score(y_true, y_pred, average="micro", zero_division=0), 3),
        "Recall":       round(recall_score(y_true, y_pred, average="micro", zero_division=0), 3),
        "Hamming loss": round(hamming_loss(y_true, y_pred), 3),
    }

summary = pd.DataFrame([
    overall_metrics(y_true, y_baseline, "TF-IDF baseline"),
    overall_metrics(y_true, y_bert,     "BERT + LoRA"),
]).set_index("Model")

print(summary.to_string())

## Per-label F1

In [ ]:
f1_baseline = f1_score(y_true, y_baseline, average=None, zero_division=0)
f1_bert     = f1_score(y_true, y_bert,     average=None, zero_division=0)

per_label = pd.DataFrame({
    "Label":           LABEL_NAMES,
    "TF-IDF baseline": f1_baseline.round(3),
    "BERT + LoRA":     f1_bert.round(3),
    "Delta":           (f1_bert - f1_baseline).round(3),
}).set_index("Label")

print(per_label.to_string())

## Per-label F1 bar chart

In [ ]:
x     = np.arange(len(LABEL_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, f1_baseline, width, label="TF-IDF baseline", color="steelblue")
bars2 = ax.bar(x + width/2, f1_bert,     width, label="BERT + LoRA",     color="darkorange")

ax.set_xlabel("Label")
ax.set_ylabel("F1 score")
ax.set_title("Per-label F1: TF-IDF baseline vs BERT + LoRA (test set)")
ax.set_xticks(x)
ax.set_xticklabels(LABEL_NAMES)
ax.set_ylim(0, 1.0)
ax.legend()
ax.bar_label(bars1, fmt="%.2f", padding=3, fontsize=8)
ax.bar_label(bars2, fmt="%.2f", padding=3, fontsize=8)

plt.tight_layout()
plt.savefig("../results/bc3_comparison_f1.png", dpi=150)
plt.show()
print("Saved: results/bc3_comparison_f1.png")

## Interpretation

The chart reveals a consistent pattern: BERT+LoRA outperforms the TF-IDF baseline on labels that require semantic understanding, while both models perform similarly on labels with strong lexical signals.

**Request, Propose, and Commit** show the largest gains. The baseline scores 0.41, 0.15, and 0.40 respectively — weak performance driven by the absence of distinctive keywords for these categories. BERT+LoRA raises them to 0.60, 0.51, and 0.60. The improvement on Propose is the most striking: the baseline nearly fails (F1 0.15), while BERT+LoRA triples it to 0.51. These are the labels that require understanding the intent behind a sentence rather than matching surface-level words — exactly where a pre-trained language model has a structural advantage over bag-of-words representations.

**Meeting** is the only label where the baseline outperforms BERT+LoRA (0.64 vs 0.59). This is expected: Meeting emails tend to contain clear lexical markers ("meeting", "schedule", "availability", "call") that TF-IDF captures reliably without needing deeper language understanding.

**Subjective** is high for both models (0.86 vs 0.84) and nearly tied. As the most frequent label in the dataset (194 of 261 emails), both classifiers learn it well regardless of their architecture.

**Informative** scores 0.00 for both models. This is not a modelling failure — there are no Informative-labelled emails in the test set, so neither model can be evaluated on this category. It reflects the small size and skewed distribution of the BC3 corpus.

Overall, BERT+LoRA achieves a higher Micro F1 (0.637 vs 0.560) and a substantially higher Macro F1 (0.525 vs 0.410). The Macro F1 gap is more meaningful here: it reflects that BERT+LoRA handles the rarer, harder-to-detect labels (Propose, Commit, Request) much better than the baseline, which effectively ignores them.